# Assignment 3: Hurricane intensity classification and wind speed regression from storm records

In Assignment 2 you cleaned the IBTrACS tropical cyclone archive. Now you will model it,
using both halves of this week's material on the same dataset: **regression** to predict a
continuous intensity, and **classification** to predict a storm's category.

IBTrACS is the World Meteorological Organization's official archive of global tropical
cyclone tracks, assembled from every regional forecast centre. It is real operational data,
which means it is messy: reporting practices differ by basin and have changed over time.

Answer each numbered question in the empty cell below it.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (mean_squared_error, r2_score, classification_report,
                             ConfusionMatrixDisplay, roc_auc_score)

## Load and aggregate the data

The same source and aggregation you used in Assignment 2. This takes a few seconds: the
full archive is a large file.

In [ ]:
url = ('https://www.ncei.noaa.gov/data/international-best-track-archive-for-climate-'
       'stewardship-ibtracs/v04r01/access/csv/ibtracs.ALL.list.v04r01.csv')

df = pd.read_csv(url, parse_dates=['ISO_TIME'], usecols=range(12), skiprows=[1],
                 na_values=[' ', 'NOT_NAMED'], low_memory=False)

# One row per named storm, summarizing its track.
storms = df.groupby("NAME").agg(
    MAX_WIND=('WMO_WIND', 'max'),
    MIN_PRES=('WMO_PRES', 'min'),
    MEAN_LAT=('LAT', 'mean'),
    MEAN_LON=('LON', 'mean'),
    SEASON=('SEASON', 'first'),
    BASIN=('BASIN', 'first'),
).dropna()

print(f"{len(storms)} named storms with complete records")
storms.head()

The **Saffir-Simpson scale** classifies storms by maximum sustained wind in knots:

| Category | Wind (kt) |
| --- | --- |
| Tropical storm | 34–63 |
| 1 | 64–82 |
| 2 | 83–95 |
| 3 | 96–112 |
| 4 | 113–136 |
| 5 | 137+ |

Categories 3 and above are **major hurricanes**.

In [ ]:
storms["CATEGORY"] = np.select(
    [storms.MAX_WIND >= 137, storms.MAX_WIND >= 113, storms.MAX_WIND >= 96,
     storms.MAX_WIND >= 83,  storms.MAX_WIND >= 64],
    [5, 4, 3, 2, 1], default=0)          # 0 = tropical storm or weaker

storms["MAJOR"] = (storms.MAX_WIND >= 96).astype(int)

print(storms["CATEGORY"].value_counts().sort_index())
print(f"\nmajor hurricanes: {storms.MAJOR.sum()} of {len(storms)}")

## Part 1: Explore

1) Plot minimum central pressure against maximum wind speed, colored by category. Describe
the relationship you see, and explain physically why it takes that form.

2) How many storms fall in each category? Is this dataset balanced? What does that imply for the classification tasks below?

3) Plot the distribution of maximum wind speed by basin. Do all basins produce the same range of intensities?

## Part 2: Linear regression: predicting intensity

4) Build a feature matrix from `MIN_PRES`, `MEAN_LAT`, `MEAN_LON` and `SEASON`, with
`MAX_WIND` as the target. Split into training and test sets with `random_state=0`.

5) Fit a `LinearRegression` and report the RMSE and $R^2$ on the test set.

6) Report the fitted coefficients. Which feature dominates, and does its sign make physical sense?

7) Plot predicted against observed wind speed for the test set, with a 1:1 line. Where does the model do worst, and is that the part of the range you would most want to get right?

8) Compare against a baseline that always predicts the mean wind speed. How much better is your model? A model that cannot beat this baseline has demonstrated nothing.

## Part 3: Logistic regression, will it be a major hurricane?

9) Using the same features, fit a `LogisticRegression` to predict `MAJOR`. Standardize the
features first. Use `make_pipeline(StandardScaler(), LogisticRegression())`. Why does
scaling matter here when it did not for linear regression?

10) Report accuracy, precision, recall and the confusion matrix on the test set.

11) Report the ROC AUC. Then compare accuracy against a classifier that always predicts the majority class. Which metric is more informative here, and why?

12) Extract the predicted probabilities and plot them against minimum pressure. Where does the model sit near 0.5, and what does that region represent physically?

## Part 4: Softmax regression, the full category scale

13) Fit a multi-class logistic regression to predict `CATEGORY` (six classes). Report the
classification report and a confusion matrix.

14) Which categories does the model confuse most? Explain why in terms of how the Saffir-Simpson categories are defined, pay attention to the width of each wind-speed bin.

15) The category is a deterministic function of `MAX_WIND`, which you predicted in Part 2. Compare two approaches: (a) classify directly, and (b) predict wind speed with regression then apply the category thresholds. Which works better, and why might that be?

## Part 5: Support vector machines

16) Fit an `SVC` with a linear kernel to the `MAJOR` classification task. Compare its
performance to logistic regression.

17) Now fit an `SVC` with an RBF kernel. Does the nonlinear boundary help? Report both.

18) Logistic regression outputs calibrated probabilities; a plain SVM outputs distances from the decision boundary. For a forecaster deciding whether to issue an evacuation order, which is more useful, and why?

*Write your answer here.*

## Part 6: Interpretation

19) Minimum pressure is by far the strongest predictor of maximum wind. Both are measurements
*of the same storm at the same time*. If your goal were to **forecast** intensity 24 hours
ahead, would pressure still be available as a feature? What does that tell you about the
difference between a model that explains and a model that predicts?

*Write your answer here.*

20) IBTrACS combines reports from different agencies whose practices have changed over decades. Name one way this could bias a model trained on the full record, and suggest how you would check for it.

*Write your answer here.*